# One-country pipeline walkthrough

Runs `backend/utils/pipeline._process_country` one step at a time, so every
intermediate is visible: the macro panel, both payloads, the raw article pool,
the stage-1 digests, the model's four ledger scores and per-article impacts, the
lint pass that watches for contradictions without correcting them, the
provenance manifest recording what the model saw, and the Top-3 that reach the
dashboard.

**Pick a country by editing `ISO2` in the cell under _Pick a country_, then Run All.**

Before the first run, install a kernel into the project venv (it is deliberately
not in `requirements.txt`, which is runtime-only):

```
.venv\Scripts\python.exe -m pip install ipykernel
```

## What this pipeline believes

Scores come from **three ledgers**, and the framework is easier to follow if you
know what each one is asking:

| Ledger | The question | Higher means |
|---|---|---|
| **Friction** | What does the state extract, and how much of it converts into capability? | a worse wedge |
| **Order-uncertainty** | Are the load-bearing rules — contracts, currency, statistics, succession — legible? | less underwritable |
| **Information** | Can the country's own instruments be trusted to measure it? | weaker instruments |
| **Edge vitality** | Is the system still learning — firms forming, failing, inventing? | *more vitality, and this one is not risk* |

Two consequences run through every step below:

- **The model's score is never edited.** There were floors, a cap and a
  sanctions gate; they are gone. `score` is the model's `score_12m` and nothing
  else. A sanctioned country keeps the score its evidence earned and gains a
  `non_investable` badge beside it. Contradictions between flags and scores are
  recorded by `utils/lint.py` and corrected by nobody.
- **Absent means absent.** Every value the model sees carries `as_of` and
  `staleness_days`; anything missing is simply not in the payload, never a
  zero and never a padded null.

Two operational things to know:

- **This writes no snapshot.** The last step builds the payload
  `data_push.upsert_snapshot` would take and prints it, but does not call it.
  The one thing that does touch Postgres is the stage-1 digest cache in Step 2b
  (`article_digest`), which is scratch data keyed by article URL — nothing the
  dashboard reads. Steps 1b and 3d *read* Postgres if `DATABASE_URL` is set and
  degrade to "no evidence from that store" if it is not.
- **It makes real network calls**, and **Steps 2b and 3 spend OpenAI credits**
  (one cheap digest call per article, then one scoring call). A country with no
  local macro panel also hits the World Bank once per indicator in Step 0, which
  is slow.

## Setup

Resolves the repo root, loads `backend/.env`, and routes the pipeline's logging
into the notebook. `force=True` on `basicConfig` is not optional — Jupyter
installs its own root handler, so a plain `basicConfig()` is silently a no-op
and no pipeline logs appear.

In [1]:
import json
import logging
import os
import pathlib
import sys

import pandas as pd
from dotenv import load_dotenv

# Repo root = the folder holding backend/main.py, so this works whether the
# kernel's cwd is backend/ or the repo root.
PROJECT_ROOT = next(
    p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (p / "backend" / "main.py").exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Same two-line load as main.py; every module reads os.getenv at call time.
load_dotenv(PROJECT_ROOT / "backend" / ".env")
load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-7s %(name)s: %(message)s",
    stream=sys.stdout,
    force=True,
)

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

print(f"project root: {PROJECT_ROOT}")
for key in ("OPENAI_API_KEY", "CRAWLBASE_TOKEN", "CRAWLBASE_JS_TOKEN"):
    print(f"  {key:<20} {'set' if os.getenv(key) else 'MISSING'}")
print(f"  {'DATABASE_URL':<20} "
      f"{'set - used by the Step 2b digest cache only' if os.getenv('DATABASE_URL') else 'MISSING - Step 2b just skips its cache'}")

project root: d:\Coding\ai-country-risk
  OPENAI_API_KEY       set
  CRAWLBASE_TOKEN      set
  CRAWLBASE_JS_TOKEN   set
  DATABASE_URL         set - used by the Step 2b digest cache only


In [2]:
from backend.utils import constants, data_retrieval, lint, metrics, provenance
from backend.utils.ai import client as ai_client
from backend.utils.ai import constants as ai_constants
from backend.utils.ai import digest_engine, langchain_llm, policy
from backend.utils.data_fetching import (
    bis_bulk_fetch, country_data_fetch, curated_loader, imf_macro_fetch, wb_series_fetch,
)
from backend.utils.data_upsert import data_push
from backend.utils.news_fetching import article_enrichment, article_ranking

# Payload window and article cap, mirrored from backend/utils/pipeline.py.
SINCE_YEAR = 2015
LOOKBACK_YEARS = 10
DELTA_HORIZONS = (1, 5)
MAX_ARTICLES = 20

print(f"pipeline modules loaded - scoring model: {ai_client.MODEL_NAME}, "
      f"digest model: {ai_client.DIGEST_MODEL_NAME}")
# The three stamps every snapshot carries, so a score can be tied to what made it.
print(f"  prompt {ai_constants.PROMPT_VERSION}   policy {policy.POLICY_VERSION}   "
      f"seed {ai_client.SEED}")
print(f"  {len(constants.INDICATOR_REGISTRY)} indicators in the registry across "
      f"{len({s['ledger'] for s in constants.INDICATOR_REGISTRY.values() if s['ledger']})} ledgers")

pipeline modules loaded - scoring model: gpt-4o-2024-08-06, digest model: gpt-4o-mini-2024-07-18
  prompt v3.1   policy p2.0-observe-only   seed 42
  31 indicators in the registry across 4 ledgers


In [3]:
# --- chart helpers -------------------------------------------------------
# No plotting library: matplotlib is not in the venv and the notebook kernel is
# deliberately kept out of requirements.txt, so a chart here must not add a
# dependency every reader has to install. Inline SVG through IPython.display
# renders in JupyterLab, VS Code and GitHub's notebook viewer alike.
from IPython.display import HTML

# Categorical slots from a validated palette, assigned by LEDGER IDENTITY and
# never cycled: a ledger keeps its hue in every chart below. (Checked for
# colour-vision separation in both modes; the light steps sit under 3:1 on the
# light surface, which is why every bar carries a visible value label.)
LEDGER_HUE = {
    "friction":    ("#2a78d6", "#3987e5"),   # slot 1 · blue
    "uncertainty": ("#eb6834", "#d95926"),   # slot 2 · orange
    "information": ("#1baf7a", "#199e70"),   # slot 3 · aqua
    "edge":        ("#eda100", "#c98500"),   # slot 4 · yellow
}
LEDGER_LABEL = {
    "friction": "friction", "uncertainty": "order-uncertainty",
    "information": "information", "edge": "edge",
}

_CSS = """<style>
.viz{color-scheme:light;--s:#fcfcfb;--ink:#0b0b0b;--ink2:#52514e;--rule:#e5e4e1;
--c1:#2a78d6;--c2:#eb6834;--c3:#1baf7a;--c4:#eda100;
font:12px/1.45 ui-sans-serif,-apple-system,"Segoe UI",sans-serif;background:var(--s);
border:1px solid var(--rule);border-radius:6px;padding:14px 16px;margin:4px 0;max-width:880px}
@media (prefers-color-scheme:dark){.viz{color-scheme:dark;--s:#1a1a19;--ink:#fff;
--ink2:#c3c2b7;--rule:#3a3a37;--c1:#3987e5;--c2:#d95926;--c3:#199e70;--c4:#c98500}}
.viz h4{margin:0;font-size:13px;font-weight:600;color:var(--ink)}
.viz .sub{margin:3px 0 12px;font-size:11px;color:var(--ink2)}
.viz .lg{display:flex;flex-wrap:wrap;gap:14px;margin-top:11px;font-size:11px;color:var(--ink2)}
.viz .lg i{width:9px;height:9px;border-radius:2px;display:inline-block;margin-right:5px;
vertical-align:-1px}
.viz .tiles{display:flex;flex-wrap:wrap;gap:10px;align-items:stretch}
.viz .tile{flex:1 1 130px;border:1px solid var(--rule);border-radius:5px;padding:9px 11px}
.viz .tile .k{font-size:10px;letter-spacing:.06em;text-transform:uppercase;color:var(--ink2)}
.viz .tile .v{font-size:19px;font-weight:600;color:var(--ink);margin-top:3px;
font-variant-numeric:tabular-nums}
.viz .tile .u{font-size:10px;color:var(--ink2);margin-top:1px}
.viz .op{flex:0 0 auto;align-self:center;font-size:16px;color:var(--ink2);padding:0 1px}
.viz text{font:11px ui-sans-serif,-apple-system,"Segoe UI",sans-serif}
</style>"""

_BAR_H, _BAND = 14, 26          # <=24px thick, and the leftover band is air
_PLOT_W, _VAL_W = 460, 74
_CH = 5.7                        # ~11px sans-serif average advance
_LBL_MAX = 268                   # past this a label is truncated, not clipped


def _fit(label):
    """Truncate a label to the column width, keeping the full text for hover."""
    room = int(_LBL_MAX / _CH)
    return label if len(label) <= room else label[:room - 1].rstrip() + "\u2026"


def hbar(rows, title, sub="", *, fmt="{:.2f}", max_value=None, footer=""):
    """Horizontal bars: rows of (label, value, ledger_key, hover_note).

    One baseline, thin marks, a 4px rounded data-end squared off at the
    baseline, and a direct value label on every bar - the labels are not
    decoration, they are what makes the chart legible when a hue sits below
    3:1 on the light surface.

    The label column is measured from the longest label rather than fixed, so a
    long indicator name pushes the plot right instead of running off the edge.
    """
    rows = [r for r in rows if r[1] is not None]
    if not rows:
        return HTML(_CSS + f'<div class="viz"><h4>{title}</h4>'
                    f'<p class="sub">no values available</p></div>')
    top = max_value or max(abs(v) for _, v, _, _ in rows) or 1.0
    lbl_w = min(_LBL_MAX, max(70, int(max(len(_fit(r[0])) for r in rows) * _CH) + 6))
    x0 = lbl_w + 10
    h = _BAND * len(rows) + 6
    out = [f'<svg width="{x0 + _PLOT_W + _VAL_W}" height="{h}" role="img">']
    # Recessive hairline baseline, one step off the surface.
    out.append(f'<line x1="{x0}" y1="2" x2="{x0}" y2="{h - 4}" '
               f'stroke="var(--rule)" stroke-width="1"/>')
    for i, (label, value, key, note) in enumerate(rows):
        y = i * _BAND + 3
        w = max(2.0, abs(value) / top * _PLOT_W)
        c = f"var(--c{list(LEDGER_HUE).index(key) + 1})"
        out.append(f'<g><title>{label}: {fmt.format(value)}{note}</title>')
        out.append(f'<text x="{lbl_w}" y="{y + 11}" text-anchor="end" '
                   f'fill="var(--ink2)">{_fit(label)}</text>')
        out.append(f'<rect x="{x0}" y="{y}" width="{w:.1f}" height="{_BAR_H}" '
                   f'rx="4" fill="{c}"/>')
        # Square the baseline end: rx rounds both, this squares the anchored one.
        out.append(f'<rect x="{x0}" y="{y}" width="{min(4.0, w):.1f}" '
                   f'height="{_BAR_H}" fill="{c}"/>')
        # Value in text ink, never the series colour.
        out.append(f'<text x="{x0 + w + 7:.1f}" y="{y + 11}" fill="var(--ink)" '
                   f'style="font-variant-numeric:tabular-nums">{fmt.format(value)}</text>')
        out.append('</g>')
    out.append('</svg>')
    seen = list(dict.fromkeys(k for _, _, k, _ in rows))
    legend = ""
    if len(seen) > 1:      # one series needs no legend box; the title names it
        legend = '<div class="lg">' + "".join(
            f'<span><i style="background:var(--c{list(LEDGER_HUE).index(k) + 1})"></i>'
            f'{LEDGER_LABEL[k]}</span>' for k in seen) + '</div>'
    return HTML(_CSS + f'<div class="viz"><h4>{title}</h4>'
                + (f'<p class="sub">{sub}</p>' if sub else "")
                + "".join(out) + legend
                + (f'<p class="sub" style="margin:9px 0 0">{footer}</p>' if footer else "")
                + '</div>')


def tiles(items, title, sub=""):
    """Stat tiles for the two-or-three numbers that are a sentence, not a chart."""
    body = []
    for it in items:
        if it == "x" or it == "=":
            body.append(f'<div class="op">{"&times;" if it == "x" else "="}</div>')
            continue
        k, v, u = it
        body.append(f'<div class="tile"><div class="k">{k}</div>'
                    f'<div class="v">{v}</div><div class="u">{u}</div></div>')
    return HTML(_CSS + f'<div class="viz"><h4>{title}</h4>'
                + (f'<p class="sub">{sub}</p>' if sub else "")
                + '<div class="tiles">' + "".join(body) + '</div></div>')


print("chart helpers ready - inline SVG, no plotting dependency")

chart helpers ready - inline SVG, no plotting dependency


## Pick a country

The table below is `constants.COUNTRY_ROSTER`. `has_panel` says whether the
macro panel is already on disk — those countries skip the slow World Bank
backfill in Step 0.

In [4]:
roster = pd.DataFrame(constants.COUNTRY_ROSTER)
roster["has_panel"] = roster["iso2"].map(
    lambda c: country_data_fetch.has_country_partition(country_data_fetch.PANEL_DIR, c)
)

print(f"{len(roster)} countries, {int(roster.has_panel.sum())} with a local macro panel")
roster.sort_values(["has_panel", "tier", "name"], ascending=[False, True, True])

48 countries, 0 with a local macro panel


,name,iso2,iso3,tier,lat,lng,has_panel
0,Australia,AU,AUS,DM,-24.6809,134.5300,False
1,Austria,AT,AUT,DM,47.6082,14.3738,False
2,Belgium,BE,BEL,DM,50.6003,4.7000,False
3,Canada,CA,CAN,DM,60.9215,-108.0070,False
4,Denmark,DK,DNK,DM,55.6761,10.5683,False
5,Finland,FI,FIN,DM,63.3000,25.6200,False
6,France,FR,FRA,DM,46.6000,2.0000,False
7,Germany,DE,DEU,DM,51.2000,10.5000,False
8,"Hong Kong SAR, China",HK,HKG,DM,22.3193,114.1694,False
9,Ireland,IE,IRL,DM,52.8000,-8.0000,False


In [5]:
ISO2 = "US"   # <-- change country here

ENTRY = next(c for c in constants.COUNTRY_ROSTER if c["iso2"] == ISO2)
NAME, ISO3 = ENTRY["name"], ENTRY["iso3"]

print(f"{NAME}  ({ISO2}/{ISO3})  tier={ENTRY['tier']}  map=({ENTRY['lat']}, {ENTRY['lng']})")

United States  (US/USA)  tier=DM  map=(39.75, -100.5)


## Step 0 — macro panel

Each country's World Bank indicators live in a Parquet partition under
`backend/data/wb_panel_wide/country_code=XX/`. If this country has none, the
real backfill runs with the roster temporarily narrowed to this one entry — the
same trick `backend/tests/live_country_check.py` uses. Expect a few minutes and
one World Bank call per indicator.

The panel is then read back through DuckDB, exactly as the pipeline reads it.

In [6]:
if country_data_fetch.has_country_partition(country_data_fetch.PANEL_DIR, ISO2):
    print(f"panel already on disk for {ISO2}")
else:
    print(f"no panel for {ISO2} - building it (slow: one World Bank call per indicator)")
    full_roster = constants.COUNTRY_ROSTER
    constants.COUNTRY_ROSTER = [ENTRY]
    try:
        country_data_fetch.backfill_missing_panels()
    finally:
        constants.COUNTRY_ROSTER = full_roster

panel = data_retrieval.query_macro_panel(ISO2)
print(f"panel: {panel.shape[0]} years x {panel.shape[1]} columns")
panel.tail(15)

no panel for US - building it (slow: one World Bank call per indicator)
2026-07-28 16:14:46,072 INFO    backend.utils.data_fetching.country_data_fetch: Backfilling 1 missing panels → ['US']
2026-07-28 16:15:10,685 INFO    backend.utils.data_fetching.political_corruption_fetch: Fetched OWID Political Corruption Index CSV: 28708 rows.
2026-07-28 16:15:10,740 INFO    backend.utils.data_fetching.country_data_fetch: [US] Wrote panel with 237 years × 9 indicators.
panel: 26 years x 11 columns


,year,INFLATION,UNEMPLOYMENT,FDI_PCT_GDP,POL_STABILITY,RULE_OF_LAW,GINI_INDEX,GDP_PC_GROWTH,INT_PAYM_PCT_REV,POL_CORRUPTION,country_code
11,2011,3.156842,8.949,1.689112,0.624543,1.363273,41.2,0.762796,12.511076,0.054,US
12,2012,2.069337,8.069,1.540208,0.620748,1.384273,41.2,1.475706,11.664939,0.053,US
13,2013,1.464833,7.375,1.706868,0.579523,1.301608,40.9,1.348163,9.320821,0.053,US
14,2014,1.622223,6.168,1.430339,0.668123,1.341257,41.7,1.710945,9.411366,0.053,US
15,2015,0.118627,5.280,2.795482,0.665879,1.377017,41.5,2.127411,8.812689,0.053,US
16,2016,1.261583,4.869,2.522681,0.327915,1.397158,41.3,1.022666,9.911887,0.068,US
17,2017,2.130110,4.355,1.941776,0.110129,1.320034,41.4,1.750141,9.793598,0.100,US
18,2018,2.442583,3.896,1.039454,0.290496,1.172107,41.8,2.364442,12.263408,0.097,US
19,2019,1.812210,3.669,1.466965,0.054637,1.123517,41.9,2.056766,12.792787,0.093,US
20,2020,1.233584,8.055,0.641236,-0.146807,1.004381,40.0,-2.480600,10.904342,0.085,US


## Step 1 — the panel payload

`prepare_llm_payload_pretty` compresses the panel into per-indicator latest
values, 1- and 5-year changes, and the last 10 observations.
`_meta.generated_at` is the timestamp that becomes the snapshot's `as_of`.

**The scoring model no longer reads this payload.** It stayed because it has a
second job: `data_push.upsert_snapshot` reads its `indicators` and
`_meta.units` to write the `indicator` and `yearly_value` tables the front-end's
indicator pane queries, and `provenance.macro_vintages` reads its `series`.
Reshaping it into ledgers would have broken both, silently, in the front-end.
What the model reads is built separately in Step 1b.

In [7]:
payload = data_retrieval.prepare_llm_payload_pretty(
    country_iso=ISO2,
    indicators=constants.ALL_INDICATORS,
    since=SINCE_YEAR,
    lookback=LOOKBACK_YEARS,
    deltas=DELTA_HORIZONS,
)

units = payload["_meta"]["units"]
indicators = pd.DataFrame([
    {
        "indicator": name,
        "unit": units.get(name, ""),
        "latest": v["latest"],
        "\u03941y": v["\u03941y"],
        "\u03945y": v["\u03945y"],
        "years": len(v["series"]),
    }
    for name, v in payload["indicators"].items()
]).set_index("indicator")

print(f"latest_year={payload['latest_year']}  generated_at={payload['_meta']['generated_at']}")
indicators

latest_year=2025  generated_at=2026-07-28T20:15Z


,unit,latest,Δ1y,Δ5y,years
indicator,,,,,
Inflation (% y/y),% y/y,2.95,-1.167,1.137,10
Unemployment (% labour force),%,4.20,0.176,-3.857,10
FDI inflow (% GDP),% GDP,1.30,0.288,0.660,10
Political stability (z-score),z-score,-0.10,0.098,-0.150,10
Rule of law (z-score),z-score,0.96,0.001,-0.162,10
Income inequality (Gini),index,41.80,0.000,-0.100,10
GDP per-capita growth (% y/y),% y/y,1.63,-0.182,4.110,10
Interest payments (% revenue),% revenue,20.29,2.272,7.498,10
"Political corruption index (0–1, higher = more corrupt)",index (0–1),0.06,0.009,-0.025,10


## Step 1b — the evidence payload (the three ledgers)

`build_evidence_payload` is what the scorer actually receives. It merges three
stores — the annual parquet panel, the monthly latest-print table, and the
generic `indicator_series` store — and for each indicator keeps the copy whose
**period is most recent**, breaking ties toward the copy we learned latest.

That resolution is the point of the whole rebuild. Before it, the model scored
Argentina's inflation on a World Bank annual average up to two years stale while
the front-end, reading the same database, showed the current monthly print.

Everything is passed in rather than read inside the builder, so it is pure and
re-runnable over history: the caller decides which snapshot of the world it
sees. Each read below degrades on its own — no database, or no curated rows,
costs the country that evidence and not its score.

In [ ]:
AS_OF = data_push.payload_as_of(payload)   # the date the snapshot would be keyed on

# Step 0 backfills the macro panel when a country has none; this does the same
# for the ledger sources. The daily run fills them for the whole roster in
# pipeline.refresh_ledger_sources(), so this only fires on a country you have
# not scored yet. Set False to score on whatever is already stored.
FETCH_MISSING_SERIES = True
FETCH_BIS = True   # ~14 MB of BIS bulk files; the only source of FX volatility


def safe(read, what):
    """Mirror the pipeline's per-store resilience: a failed read is absent, not fatal."""
    try:
        return read()
    except Exception as exc:
        print(f"  {what:<18} unavailable ({type(exc).__name__}) - degrading to absent")
        return None


def as_series_dict(rows):
    """Group upsert-shaped rows the way read_indicator_series returns them."""
    out = {}
    for r in sorted(rows, key=lambda r: (r["indicator_code"], r["period"])):
        out.setdefault(r["indicator_code"], []).append(r)
    return out


series = safe(lambda: data_push.read_indicator_series(ISO2), "indicator_series") or {}

if not series and FETCH_MISSING_SERIES:
    print(f"no stored series for {ISO2} - fetching (slow: World Bank, IMF"
          f"{', BIS bulk' if FETCH_BIS else ''})")
    fetched = []
    fetched += wb_series_fetch.fetch_country_series(ISO2, ISO3, as_of=AS_OF) or []
    fetched += imf_macro_fetch.fetch_series_rows(ISO2, ISO3, as_of=AS_OF) or []
    if FETCH_BIS:
        for code in ("BIS.POLICY.RATE", "BIS.FX.USD"):
            fetched += [r for r in bis_bulk_fetch.fetch_dataset_rows(code, as_of=AS_OF)
                        if r["country_iso2"] == ISO2]
    fetched += [r for r in safe(curated_loader.load_curated_series, "curated.csv") or []
                if r["country_iso2"] == ISO2]
    # Held in memory and handed straight to the builder, so the notebook needs
    # no database. Persisted too when there is one, so a re-run is instant.
    series = as_series_dict(fetched)
    if os.getenv("DATABASE_URL") and fetched:
        safe(lambda: data_push.upsert_indicator_series(fetched), "series upsert")

recent = safe(lambda: data_push.read_recent_indicators(ISO2), "recent_indicator") or {}

evidence = data_retrieval.build_evidence_payload(
    ISO2, as_of=AS_OF,
    panel=panel,               # the DataFrame from Step 0
    series=series, recent=recent,
    fx_regimes=constants.FX_REGIMES, elections=constants.ELECTIONS,
)

sections = {k: v for k, v in evidence.items() if k.endswith("_inputs")}
print()
print(f"as_of {AS_OF}   vintage {evidence['_meta']['vintage_scheme']}")
print(f"  indicator_series   {len(series)} indicators, "
      f"{sum(len(v) for v in series.values())} observations")
print(f"  recent_indicator   {len(recent)} latest prints")
print(f"  curated            {len(constants.FX_REGIMES)} fx regimes, "
      f"{len(constants.ELECTIONS)} election calendars")
for name, block in sections.items():
    print(f"  {name:<20} {len(block)} entries")
print(f"  computed           {len(evidence['computed'])} metrics")

missing = sorted(
    str(spec["label"]) for code, spec in constants.INDICATOR_REGISTRY.items()
    if spec["ledger"] and code not in series
    and not any(str(spec["label"]) in b for b in sections.values())
)
if missing:
    print()
    print(f"  absent from every store ({len(missing)}): {', '.join(missing[:6])}"
          f"{' ...' if len(missing) > 6 else ''}")
    print("  Absent, not zero - these simply do not appear in the payload.")

### What each ledger is carrying, and how old it is

`staleness_days` counts from the **end of the period a value describes** to
`as_of` — how old the reading is. That is a different fact from the value's
`as_of`, which is when it became known to us, and conflating the two is exactly
how a stale number passes for a current one: a 2020 Human Capital Index reading
has `as_of` = today, and stamping staleness against *that* would tell the model
a six-year-old figure is fresh.

So the bars below are the honest age of the evidence. A short bar is a reading
the model can lean on; a long one it should discount, and the prompt tells it
to.

In [9]:
rows = []
for section, key in (("friction_inputs", "friction"),
                     ("uncertainty_inputs", "uncertainty"),
                     ("information_inputs", "information"),
                     ("edge_inputs", "edge")):
    entries = [(n, e) for n, e in evidence[section].items()
               if isinstance(e, dict) and e.get("staleness_days") is not None]
    # Grouped by ledger, freshest first within each: a fixed order keeps every
    # colour pair on the chart an adjacent one.
    for name, e in sorted(entries, key=lambda kv: kv[1]["staleness_days"]):
        rows.append((name[:46], e["staleness_days"], key,
                     f" · {e['period']} ({e['freq']}) · {e['source']}"))

display(hbar(
    rows,
    f"Evidence age - {NAME} as of {AS_OF}",
    "days from the end of the period each value describes. Hover for period and source.",
    fmt="{:.0f}d",
    footer=f"{len(rows)} indicators present. Anything the registry defines but no "
           f"store has is absent from the payload entirely, never a padded null.",
))

### Is a long bar a defect, or just the calendar?

The chart above cannot say, and for the United States it misleads twice.

A World Bank annual series at 574 days is a series doing its job: the 2024 round
is the newest one that exists. The Human Capital Index at 2035 days looks far
worse, but `freq` says `A` and the index has never been annual — it arrives in
irregular rounds, 2017, 2018 and 2020, with none since. Against a triennial
cadence that bar is 1.9 cycles: one round missed, real but not the emergency its
length implies.

Normalizing by cadence then inverts the ranking. Education spending *is* annual,
and the newest US value is 2021, so 4.6 annual rounds have come and gone with
nothing new. That, not the human-capital bar beside it, is where the edge ledger
is actually blind — and the raw-age chart draws it as the shorter of the two.

The tiles roll the same ages up per ledger. Edge sits oldest for every country,
not just this one: learning outcomes and human capital are measured on multi-year
assessment cycles, and no source anywhere reports them faster.

In [10]:
from statistics import median

# How long a fresh reading is expected to stay the newest one, by frequency.
CADENCE = {"M": 30, "Q": 91, "A": 365}

# Two series declare freq "A" because their period is a year, yet neither
# republishes annually - the HCI arrives in irregular rounds (2017/2018/2020),
# PISA on a fixed 3-year cycle. Keyed by `source` because that is what a payload
# entry carries: the label is prose and the registry code never reaches the
# payload at all.
CYCLE_DAYS = {"World Bank Human Capital Project": 365 * 3, "OECD PISA": 365 * 3}

# Same section order and same freshest-first sort as the chart above, so a row
# sits in the same place in both and the ledger hues stay adjacent.
entries = []
for section, key in (("friction_inputs", "friction"),
                     ("uncertainty_inputs", "uncertainty"),
                     ("information_inputs", "information"),
                     ("edge_inputs", "edge")):
    present = [(n, e, key) for n, e in evidence[section].items()
               if isinstance(e, dict) and e.get("staleness_days") is not None]
    entries += sorted(present, key=lambda t: t[1]["staleness_days"])

behind = [
    (name[:46], e["staleness_days"] / CYCLE_DAYS.get(e["source"], CADENCE[e["freq"]]),
     key, f" · {e['staleness_days']}d · {e['period']} ({e['freq']}) · {e['source']}")
    for name, e, key in entries
]
late = sum(1 for _, cycles, _, _ in behind if cycles >= 2.0)

display(hbar(
    behind,
    f"Publication cycles behind - {NAME} as of {AS_OF}",
    "evidence age divided by how often the source republishes. Hover for the raw "
    "age, period and source.",
    fmt="{:.1f}×",
    footer=f"1.0× is a reading that arrived on schedule; 2.0× means a full cycle "
           f"came and went without one. {late} of {len(behind)} indicators sit at "
           f"2.0× or worse - those are the ones to discount, and only those.",
))

rollup = []
for key in dict.fromkeys(key for _, _, key in entries):
    ages = [e["staleness_days"] for _, e, k in entries if k == key]
    rollup.append((LEDGER_LABEL[key], f"{median(ages):.0f}d",
                   f"{len(ages)} indicators · oldest {max(ages)}d"))

display(tiles(
    rollup,
    "Median evidence age by ledger",
    "the median is the ledger's working age; the oldest is what the model has to "
    "discount inside it.",
))

### The computed block

Code does the deterministic arithmetic so the model does not have to do sums in
its head. Each of these is a pure function in `backend/utils/metrics.py`, and
any missing input yields `None` rather than a fabricated zero — so a metric
absent from this table is a statement about our evidence, not about the country.

In [11]:
computed = evidence["computed"]

# The headline measure is a sentence, not a chart: what the state takes, times
# how badly it converts, is the wedge.
take = evidence["friction_inputs"].get("Tax revenue (% GDP)", {}).get("value")
loss = computed.get("conversion_loss")
wedge = computed.get("frictional_extraction")
if wedge is not None:
    display(tiles(
        [("extraction", f"{take:.1f}%", "tax revenue, % GDP"), "x",
         ("conversion loss", f"{loss:.3f}", "0-1, from GE z-score + corruption"), "=",
         ("the wedge", f"{wedge:.2f}%", "of GDP taken and lost")],
        "Frictional extraction",
        "Judge the take by how it converts, not by its size: a large take that "
        "funds functioning courts and registries is not friction; a modest one "
        "that funds nothing is.",
    ))

# The purity claim, made checkable: conversion_loss took two published numbers
# and nothing else, so recomputing it here from the payload must land on the
# same value. No clock, no network, no roster - which is what lets the whole
# layer be re-run over history for free.
ge = evidence["friction_inputs"].get("Government effectiveness (z-score)", {}).get("value")
corr = evidence["friction_inputs"].get(
    "Political corruption index (0\u20131, higher = more corrupt)", {}).get("value")
if loss is not None:
    again = metrics.conversion_loss(ge, corr)
    print(f"conversion_loss({ge}, {corr}) = {again}   stored {loss}   "
          f"{'match' if again == loss else 'MISMATCH'}")

flat = {}
for k, v in computed.items():
    if isinstance(v, dict):
        flat.update({f"{k}.{kk}": vv for kk, vv in v.items()})
    else:
        flat[k] = v
display(pd.Series(flat, name="value").to_frame())

flag = evidence["uncertainty_inputs"]["suppressed_vol_flag"]
print(f"suppressed_vol_flag  {flag['value']!s:<6} "
      f"regime={flag['regime']!s:<8} fx_vol={flag['fx_volatility_24m']!s:<10} "
      f"reserves_6m={flag['reserves_trend_6m']}")
print("  null is not False - it means one of the three inputs is unavailable.")
print("  When true, the prompt tells the model to read measured calm as evidence")
print("  AGAINST the country: a defended peg draining reserves is fuel load.")

conversion_loss(1.36, 0.06) = 0.144   stored 0.144   match


,value
conversion_loss,0.144
frictional_extraction,1.5508
doom_loop.burden_5y_delta,0.8847
doom_loop.conversion_quality_5y_delta,-0.0064
doom_loop.burden_up_quality_down,True
monetary_dilution,4.4146
real_policy_rate,0.0936
cpi_volatility_36m,0.458671
fx_volatility_24m,0.0
precommitted_share.value,20.2909


suppressed_vol_flag  None   regime=None     fx_vol=0.0        reserves_6m=None
  null is not False - it means one of the three inputs is unavailable.
  When true, the prompt tells the model to read measured calm as evidence
  AGAINST the country: a defended peg draining reserves is fuel load.


## Step 2 — news

Four Google News queries per country (broad, government, economic, security),
de-duplicated by URL and scored by the keyword heuristic in
`article_ranking.score_relevance`. Anything under 0.3 is dropped as noise —
unless that leaves fewer than 3 articles, in which case the bar is relaxed
rather than returning an empty pane.

In [12]:
items = article_enrichment.fetch_relevant_news(NAME, max_articles=MAX_ARTICLES)

print(f"{len(items)} articles after de-dupe, scoring, and the relevance cut")
pd.DataFrame([
    {
        "relevance": it.get("relevance_score"),
        "published": it.get("published"),
        "source": it.get("source"),
        "title": it.get("title"),
    }
    for it in items
])

2026-07-28 16:15:13,165 INFO    backend.utils.news_fetching.source_filter: Loaded 1 blocked news source(s).
2026-07-28 16:15:16,783 INFO    httpx: HTTP Request: GET https://www.cnn.com/2026/07/28/world/live-news/iran-trump-news "HTTP/1.1 200 OK"
2026-07-28 16:15:17,042 INFO    httpx: HTTP Request: GET https://www.aljazeera.com/news/2026/7/28/new-poll-shows-confusion-over-democratic-socialists-of-america-ideals "HTTP/1.1 200 OK"
2026-07-28 16:15:17,173 INFO    httpx: HTTP Request: GET https://foreignpolicy.com/2026/07/28/washingtons-reset-serbia-mistake-vucic-autocracy/ "HTTP/1.1 200 OK"
2026-07-28 16:15:17,352 INFO    httpx: HTTP Request: GET https://www.nytimes.com/2026/07/28/world/middleeast/iran-war-us-trump.html "HTTP/1.1 403 Forbidden"
2026-07-28 16:15:17,449 INFO    httpx: HTTP Request: GET https://www.bbc.com/news/articles/c87nj3w9gxjo "HTTP/1.1 200 OK"
2026-07-28 16:15:17,532 INFO    httpx: HTTP Request: GET https://time.com/article/2026/07/28/us-walks-out-protest-france-united

,relevance,published,source,title
0,1.00,2026-07-28T13:30:35Z,Breaking Defense,Time to decide: A critical moment in Japan-US missile defense cooperation - ...
1,1.00,2026-07-28T12:37:57Z,Economy Middle East,Dollar holds four-week high at 101.425 as Fed hike odds near 40 percent - Ec...
2,1.00,2026-07-27T12:34:47Z,Brookings,Why can’t the United States end allies’ wars? - Brookings
3,1.00,2026-07-26T15:11:03Z,Yahoo Finance UK,Barclays Expects Earnings and Central Bank Decisions to Set the Market Tone ...
4,1.00,2026-07-17T07:00:00Z,Peter G. Peterson Foundation,Budget Basics: National Defense - Peter G. Peterson Foundation
5,0.96,2026-07-28T16:39:00Z,NBC News,Netanyahu joins Trump for high-stakes meeting at the White House - NBC News
6,0.96,2026-07-28T05:55:40Z,Economy Middle East,"Gold falls 0.81 percent to $4,041 as dollar strengthens ahead of Fed decisio..."
7,0.95,2026-07-28T20:01:34Z,Foreign Policy,Washington’s Reset With Serbia Is a Mistake - Foreign Policy
8,0.95,2026-07-28T03:46:35Z,FXStreet,Japanese Yen flattens against US Dollar while Fed’s policy takes centre stag...
9,0.95,2026-07-26T15:17:00Z,OurQuadCities,United States could be in too deep in war with Iran to get out quickly - Our...


### Resolve and enrich

Google News links are redirect wrappers; these get unwrapped to real publisher
URLs, denylisted sources are dropped, and one GET per article recovers a
summary, body text, and thumbnail. Each survivor then gets the stable id
(`a1`, `a2`, …) the model refers back to.

In [13]:
before_count = len(items)
items = article_enrichment.resolve_and_enrich(items, ISO2)

# Stable ids for the model to cite back (pipeline.py:128-129).
for i, it in enumerate(items, start=1):
    it["id"] = f"a{i}"

print(f"{len(items)}/{before_count} survived the source denylist")
pd.DataFrame([
    {
        "id": it["id"],
        "source": it.get("source"),
        "words": len((it.get("content") or it.get("text") or "").split()),
        "image": bool(it.get("image")),
        "title": it.get("title"),
    }
    for it in items
]).set_index("id")

20/20 survived the source denylist


,source,words,image,title
id,,,,
a1,Breaking Defense,1488,True,Time to decide: A critical moment in Japan-US missile defense cooperation - ...
a2,Economy Middle East,1555,True,Dollar holds four-week high at 101.425 as Fed hike odds near 40 percent - Ec...
a3,Brookings,1551,True,Why can’t the United States end allies’ wars? - Brookings
a4,Yahoo Finance UK,298,True,Barclays Expects Earnings and Central Bank Decisions to Set the Market Tone ...
a5,Peter G. Peterson Foundation,587,True,Budget Basics: National Defense - Peter G. Peterson Foundation
a6,NBC News,1148,True,Netanyahu joins Trump for high-stakes meeting at the White House - NBC News
a7,Economy Middle East,1275,True,"Gold falls 0.81 percent to $4,041 as dollar strengthens ahead of Fed decisio..."
a8,Foreign Policy,1248,True,Washington’s Reset With Serbia Is a Mistake - Foreign Policy
a9,FXStreet,866,True,Japanese Yen flattens against US Dollar while Fed’s policy takes centre stag...


## Step 2b — stage-1 digests

The scorer never sees raw article bodies. A cheap model (`gpt-4o-mini`) reads
each article's **full text** and returns a strict-JSON factual extraction plus a
0-100 `stage1_severity`. Every digest reaches the scorer; only the three
highest-severity articles are pasted in full.

Digests are cached in Postgres by `(country, as_of, url)` plus a hash of the
digested text, so re-running this cell on the same day costs nothing. `as_of` is
read from the payload the same way `upsert_snapshot` reads it, so the cache key
matches the snapshot's.

Nothing here raises: a per-article failure leaves `digest=None` (that article
degrades to its title and summary in the prompt), and a cache read/write problem
just means "no cache".

In [14]:
AS_OF = data_push.payload_as_of(payload)   # the date upsert_snapshot would key on
items = digest_engine.digest_articles(items, country_display=NAME, iso2=ISO2, as_of=AS_OF)
fulltext_ids = digest_engine.select_fulltext_ids(items)

print(f"as_of={AS_OF}   full text goes to the scorer for: {fulltext_ids}")
pd.DataFrame([
    {
        "id": it["id"],
        "severity": it.get("stage1_severity"),
        "full_text": it["id"] in fulltext_ids,
        "about_country": (it.get("digest") or {}).get("directly_about_country"),
        "what_happened": (it.get("digest") or {}).get("what_happened"),
    }
    for it in items
]).sort_values("severity", ascending=False).set_index("id")

2026-07-28 16:16:19,691 INFO    httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-28 16:16:19,840 INFO    httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-28 16:16:19,983 INFO    httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-28 16:16:20,240 INFO    httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-28 16:16:20,289 INFO    httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-28 16:16:20,699 INFO    httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-28 16:16:20,721 INFO    httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-28 16:16:21,355 INFO    httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-28 16:16:21,622 INFO    httpx: HTTP Requ

,severity,full_text,about_country,what_happened
id,,,,
a10,85.0,True,True,The war with Iran has escalated with daily attacks by both countries on high...
a18,85.0,True,True,Israeli Prime Minister Benjamin Netanyahu is expected to present intelligenc...
a12,85.0,True,False,"Iranian ballistic missiles struck Jordan, killing three American service mem..."
a1,60.0,False,True,"Japan faces critical decisions regarding its missile defense modernization, ..."
a17,60.0,False,True,Trump urged caution while Netanyahu pushed for faster action on Iran during ...
a6,60.0,False,True,President Donald Trump met with Israeli Prime Minister Benjamin Netanyahu at...
a3,60.0,False,True,The United States faces challenges in influencing its partners' decisions on...
a2,40.0,False,True,The U.S. dollar remained close to its strongest level in four weeks as inves...
a4,40.0,False,True,"Barclays warned that rising oil prices, higher bond yields, and renewed infl..."


### What the mini model actually did

Each stage-1 call is deliberately narrow: a few thousand characters of one
article go in, a few hundred characters of strict JSON come out. The mini model
is an **extraction engine**, not an analyst — it is told to use only the text in
front of it and to write `"not stated"` rather than fill a gap from outside
knowledge. It never sees the macro payload, the other articles, or the scoring
rubric, and it never produces a risk score.

The one judgement it makes is `stage1_severity`, and that number is used for
exactly one thing: choosing which three articles the scorer reads in full.

In [15]:
digested = [it for it in items if it.get("digest")]
chars_in = sum(len(digest_engine.article_input_text(it)) for it in digested)
chars_out = sum(len(json.dumps(it["digest"], ensure_ascii=False)) for it in digested)

print(f"MINI MODEL  ({ai_client.DIGEST_MODEL_NAME})")
print(f"  {len(digested)} calls, one per article")
print(f"  read    {chars_in:>8,} chars of article text")
print(f"  wrote   {chars_out:>8,} chars of JSON   "
      f"({chars_out / max(chars_in, 1):.0%} of what it read)")

# The highest-severity article, end to end.
focus = max(digested, key=lambda it: it.get("stage1_severity") or 0.0)
sent = digest_engine.article_input_text(focus)
returned = json.dumps(focus["digest"], indent=2, ensure_ascii=False)

print(f"\n{'=' * 78}\nONE CALL IN FULL - article {focus['id']}: {focus.get('title', '')[:60]}\n{'=' * 78}")
print(f"\n--- IN: the article text it read ({len(sent):,} chars, first 800 shown) ---\n")
print(sent[:800] + ("..." if len(sent) > 800 else ""))
print(f"\n--- OUT: the entire digest it returned ({len(returned):,} chars) ---\n")
print(returned)

MINI MODEL  (gpt-4o-mini-2024-07-18)
  20 calls, one per article
  read     143,989 chars of article text
  wrote     11,667 chars of JSON   (8% of what it read)

ONE CALL IN FULL - article a10: United States could be in too deep in war with Iran to get o

--- IN: the article text it read (1,965 chars, first 800 shown) ---

We’re five months into the war with Iran. The brief respite we saw in June escalated this month with now daily attacks carried out by both countries on high-profile targets. The memorandum of understanding intended to lead to peace talks seems meaningless now. Wars are not popular in general. This one is especially unpopular with an Ipsos poll indicating 58 percent to 69 percent of the American public opposes it.
This week secretary of defense Pete Hegseth asked Congress for more money to fight the war after saying four months ago Iran’s military was destroyed. Whether the objective is regime change, preventing Iran from getting a nuclear weapon or reopening the Str

## Step 3 — LLM scoring 💸

One structured-output call returning four ledger scores, two horizon scores,
condition flags and per-article impacts — given the three-ledger evidence,
**every** article's digest, and the full text of the three highest-severity ones.

It never raises. Without `OPENAI_API_KEY`, or on a network or parse failure, it
returns `score=None` and no article scores — the rest of the notebook still
runs, the tables are just empty. If Step 2b produced no digests at all, it logs
an ERROR and falls back to the title-and-summary prompt rather than skipping the
country.

Nothing downstream will change what it returns.

### What the scoring model actually receives

`country_llm_score` assembles three blocks into `AI_PROMPT_V3`. This cell builds
them with the same internal helpers the function calls, so what prints here is
what gets sent:

- **EVIDENCE_JSON** — the three-ledger payload from Step 1b, every value stamped
  with its period, frequency, source and staleness.
- **ARTICLES_JSON** — every article, as `{id, source, published_at, title,
  digest, stage1_severity}`. No article bodies. An article whose digest failed
  appears in the old `summary` shape instead, which is how you spot a degraded one.
- **FULL_TEXT** — only the three articles Step 2b ranked highest, verbatim, each
  capped at `_MAX_FULLTEXT_CHARS` (12,000).

So the scorer sees *breadth* from the digests and *depth* on the few articles
that earned it. Unlike the mini model it holds the rubric — the ledger
definitions, bands, calibration anchors, the three-door event test — and it is
the only model that outputs a risk score.

What it no longer holds, and what nothing else holds either, is the
**enforcement**. The prompt says so in as many words: *nothing downstream will
alter your scores, and you must not adjust them to anticipate any rule.* A model
that pre-applies a floor it expects makes its own judgement unrecoverable.

In [16]:
evidence_json = json.dumps(evidence, ensure_ascii=False)
digests_json = langchain_llm._digests_to_json(items)          # the prompt's builders,
fulltext = langchain_llm._fulltext_block(items, fulltext_ids)  # not a re-implementation

print(f"SCORING MODEL  ({ai_client.MODEL_NAME})  -  1 call")
print(f"  EVIDENCE_JSON        {len(evidence_json):>8,} chars   three-ledger evidence")
print(f"  ARTICLES_JSON        {len(digests_json):>8,} chars   all {len(items)} articles, digests only")
print(f"  FULL_TEXT            {len(fulltext):>8,} chars   {len(fulltext_ids)} articles verbatim: {', '.join(fulltext_ids)}")

# For contrast: what every article's full text would have cost.
all_text = sum(len(digest_engine.article_input_text(it)) for it in items)
print(f"\n  (all {len(items)} bodies in full would be {all_text:,} chars - "
      f"the digests carry them in {len(digests_json):,})")

print(f"\n{'=' * 78}\nARTICLES_JSON - one entry, {len(items)} of these\n{'=' * 78}\n")
print(json.dumps(json.loads(digests_json)[0], indent=2, ensure_ascii=False))
print(f"\n{'=' * 78}\nFULL_TEXT - first 700 chars\n{'=' * 78}\n")
print(fulltext[:700] + ("..." if len(fulltext) > 700 else ""))

print(f"\n{'=' * 78}\nDIVISION OF LABOR\n{'=' * 78}")
pd.DataFrame([
    {
        "stage": "1 · digest",
        "model": ai_client.DIGEST_MODEL_NAME,
        "calls": len(digested),
        "reads": "one article's full text, nothing else",
        "returns": "facts + stage1_severity 0-100",
        "decides": "which 3 articles stage 2 reads in full",
    },
    {
        "stage": "2 · score",
        "model": ai_client.MODEL_NAME,
        "calls": 1,
        "reads": "3-ledger evidence + all digests + 3 full texts",
        "returns": "both horizons, 4 ledger scores, flags, per-article impact",
        "decides": "the risk score, and the dashboard Top-3",
    },
]).set_index("stage")

SCORING MODEL  (gpt-4o-2024-08-06)  -  1 call
  EVIDENCE_JSON           6,658 chars   three-ledger evidence
  ARTICLES_JSON          15,791 chars   all 20 articles, digests only
  FULL_TEXT              23,318 chars   3 articles verbatim: a10, a12, a18

  (all 20 bodies in full would be 143,989 chars - the digests carry them in 15,791)

ARTICLES_JSON - one entry, 20 of these

{
  "id": "a1",
  "source": "Breaking Defense",
  "published_at": "2026-07-28",
  "title": "Time to decide: A critical moment in Japan-US missile defense cooperation - Breaking Defense",
  "digest": {
    "what_happened": "Japan faces critical decisions regarding its missile defense modernization, particularly concerning the acquisition of IBCS and LTAMDS.",
    "actors": "Japan's government, specifically Prime Minister Takaichi Sanae and Defense Minister Shinjirō Koizumi, are making decisions about missile defense modernization in cooperation with the United States.",
    "numbers": "Japan's defense expenditures 

,model,calls,reads,returns,decides
stage,,,,,
1 · digest,gpt-4o-mini-2024-07-18,20,"one article's full text, nothing else",facts + stage1_severity 0-100,which 3 articles stage 2 reads in full
2 · score,gpt-4o-2024-08-06,1,3-ledger evidence + all digests + 3 full texts,"both horizons, 4 ledger scores, flags, per-article impact","the risk score, and the dashboard Top-3"


In [17]:
llm_output = langchain_llm.country_llm_score(
    country_display=NAME,
    payload=evidence,          # the three-ledger payload, not the panel one
    articles=items,
    as_of=AS_OF,
    fulltext_ids=fulltext_ids,
)

print(f"score: {llm_output.get('score')}   (the model's score_12m, unedited)\n")
print(llm_output.get("bullet_summary"))

2026-07-28 16:16:27,544 INFO    httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
score: 0.62   (the model's score_12m, unedited)

The U.S. faces moderate friction due to rising interest payments and a low tax revenue to GDP ratio. Order uncertainty is moderate, influenced by inflation and political stability concerns. Information capacity is strong, with reliable statistics. The ongoing U.S.-Iran war significantly impacts risk, raising short-term investor concerns. Edge vitality is moderate, reflecting adaptive capacity. Coverage is robust but some indicators are stale.


In [18]:
ledgers = llm_output.get("ledger_scores") or {}
evid = llm_output.get("subscore_evidence") or {}

display(hbar(
    [("friction", ledgers.get("friction"), "friction", " · higher = a worse wedge"),
     ("order-uncertainty", ledgers.get("order_uncertainty"), "uncertainty",
      " · higher = less underwritable"),
     ("information", ledgers.get("information_capacity"), "information",
      " · higher = weaker instruments"),
     ("edge vitality", ledgers.get("edge_vitality"), "edge",
      " · NOT risk - higher = more vitality")],
    f"Ledger scores - {NAME}",
    "0-1. The first three are risk and point the same way. The fourth is not, "
    "and may never raise a horizon score.",
    max_value=1.0,
    footer="Edge vitality is reported, never penalised: churn, startup formation "
           "AND failure, and human-capital formation are the system "
           "learning. A country where "
           "nothing is created and nothing fails is not stable, it is inert.",
))

# Two horizons, scored independently. Friction sets the level, order-uncertainty
# the width around it, information the drift between now and the horizon.
display(hbar(
    [("score_3m", llm_output.get("score_3m"), "uncertainty", " · 3-month horizon"),
     ("score_12m", llm_output.get("score"), "uncertainty", " · 12-month horizon")],
    "Horizon scores",
    "stored in risk_snapshot.score and .score_3m, exactly as returned",
    max_value=1.0,
))

display(pd.Series({k: ", ".join(v) for k, v in evid.items()},
                  name="cited evidence").to_frame())
pd.DataFrame(llm_output.get("news_article_scores") or [])

,cited evidence
friction,"Tax revenue (% GDP), Interest payments (% revenue), Government effectiveness..."
order_uncertainty,"Inflation (% y/y), Political stability (z-score), Rule of law (z-score), GDP..."
information_capacity,Statistical performance (0–100)


,id,impact,topic_group
0,a10,0.85,us_iran_war
1,a12,0.85,us_iran_war
2,a18,0.85,us_iran_war
3,a1,0.60,us_japan_missile_defense
4,a2,0.40,us_fed_rate_decision
5,a3,0.60,us_foreign_policy
6,a4,0.40,us_inflation_oil_prices
7,a5,0.25,us_defense_spending
8,a6,0.60,us_israel_relations
9,a7,0.40,us_fed_rate_decision


## Step 3b — observe, don't enforce

This step used to be the enforcement layer. `ai/policy.py` ran the
condition-flag floors, the inflation tiers read off the measured CPI, the
political-stability cap, and last the sanctions gate that forced a score to
`1.0`. All of it is deleted, along with `risk_policy.yaml`.

The distinction that survived is between a **floor** and a **badge**:

- A floor was a claim about *risk*, expressed by overwriting the model's
  judgement with a number from a YAML file. Afterwards nobody could tell which
  half of a stored score came from the model and which from the rule — which
  made both unauditable.
- A badge is a claim about *law*. Whether US persons may lawfully hold a
  country's securities is a fact about the sanctions regime, not an opinion
  about that country's risk, and it belongs next to the score rather than
  inside it.

So `assess_investability` returns `non_investable` and the triggering rule, and
touches no score. A sanctioned country keeps whatever its evidence earned it —
which is also the only way its score series stays readable across the date a
sanctions regime starts or ends. `raw_score_12m` is still written and is now
equal to `score`, so anything reading the raw columns survived the change.

In [19]:
print(f"model {llm_output.get('model_id')}   prompt {llm_output.get('prompt_version')}   "
      f"policy {llm_output.get('policy_version')}")
print(f"non_investable     {llm_output.get('non_investable')}")
print(f"legal_gate         {llm_output.get('legal_gate') or '(unrestricted at this as_of)'}")
print(f"applied_rules      {llm_output.get('applied_rules') or '(none)'}"
      "   <- observations only; nothing here moved a number")
print(f"condition_flags    {llm_output.get('condition_flags')}")
print(f"evidence_coverage  {llm_output.get('evidence_coverage')}"
      "   (how completely the evidence captures the situation)\n")

# The audit trail that used to show what policy moved. It now shows that nothing
# did - raw and stored are the same number by construction, and this table is
# here to keep proving it.
audit = pd.DataFrame([
    {"field": "score_12m", "model_said": llm_output.get("raw_score_12m"),
     "stored": llm_output.get("score")},
    {"field": "score_3m", "model_said": llm_output.get("raw_score_3m"),
     "stored": llm_output.get("score_3m")},
]).set_index("field")
audit["moved_by"] = (pd.to_numeric(audit["stored"], errors="coerce")
                     - pd.to_numeric(audit["model_said"], errors="coerce")).round(6)
display(audit)
assert (audit["moved_by"].fillna(0) == 0).all(), "a score was edited - that should be impossible"
print("moved_by is 0 everywhere, and the assert above is the guarantee.")

model gpt-4o-2024-08-06   prompt v3.1   policy p2.0-observe-only
non_investable     False
legal_gate         (unrestricted at this as_of)
applied_rules      (none)   <- observations only; nothing here moved a number
condition_flags    {'war_on_territory': False, 'internal_conflict_level': 'none', 'emergency_rule': False, 'sovereign_stress': False}
evidence_coverage  0.75   (how completely the evidence captures the situation)



,model_said,stored,moved_by
field,,,
score_12m,0.62,0.62,0.0
score_3m,0.58,0.58,0.0


moved_by is 0 everywhere, and the assert above is the guarantee.


## Step 3c — lint: contradictions, noticed but not corrected

Removing enforcement did not remove the question it was answering badly: what
happens when the model flags an active war and then scores the country a 44?

The old answer was to overwrite the 44. `utils/lint.py` writes both down
instead, next to each other, and lets a human look. Its thresholds are advisory
tripwires, not policy — they encode no view about what a score *should* be. A
war flag beside 68 is a judgement call; beside 44 it is a contradiction, and the
gap between those two numbers is exactly the room the model is allowed to use.

Findings go to the log and to the `risk_lint` table. Nothing reads them back to
change a score, and the pipeline does not block on them.

In [20]:
findings = lint.check(
    country_iso2=ISO2,
    as_of=AS_OF,
    condition_flags=llm_output.get("condition_flags"),
    # lint's tripwires are on the model's 0-100 grid; stored values are 0-1.
    score_3m=round((llm_output.get("score_3m") or 0) * 100) or None,
    score_12m=round((llm_output.get("score") or 0) * 100) or None,
    ledger_scores={k: round(v * 100) for k, v in (llm_output.get("ledger_scores") or {}).items()
                   if v is not None},
    suppressed_vol_flag=evidence["uncertainty_inputs"]["suppressed_vol_flag"]["value"],
    non_investable=bool(llm_output.get("non_investable")),
)

print(f"tripwires:  war<{lint.WAR_SCORE_FLOOR}  "
      f"sovereign_stress<{lint.SOVEREIGN_STRESS_SCORE_FLOOR}  "
      f"suppressed_calm<{lint.SUPPRESSED_CALM_UNCERTAINTY_FLOOR}  (0-100, advisory)\n")
if findings:
    lint.log_findings(findings)
    display(pd.DataFrame(findings)[["rule", "detail"]])
else:
    print("no findings - the model's flags and its scores agree.")
print("\nThis notebook does not write them; the pipeline calls "
      "data_push.upsert_lint_findings(findings) here.")

tripwires:  war<70  sovereign_stress<55  suppressed_calm<40  (0-100, advisory)

no findings - the model's flags and its scores agree.

This notebook does not write them; the pipeline calls data_push.upsert_lint_findings(findings) here.


## Step 3d — provenance: what the model saw

A stored score used to be unreproducible: the row said 0.62 and nothing else —
not which articles the scorer read, not whether it read them in full, not which
vintage of the macro panel it was reasoning over. `utils/provenance.py` builds
the record that closes that gap, and `upsert_snapshot` writes it to
`risk_snapshot.input_manifest`.

Per article it stores two hashes, because "what we held" and "what the model
read" are different questions and the gap between them is the interesting one.
`content_sha256` covers the body we downloaded; `prompt_text_sha256` covers the
exact prompt entry, taken from the same `prompt_entries` the prompt itself was
serialized from — so a re-run whose hashes match saw byte-identical evidence, and
one whose hashes differ can be told apart from a model that simply changed its
mind. `in_prompt` and `in_fulltext` record breadth versus depth; an article that
was fetched but never sent is listed too, because "we had this and did not use
it" is also a fact.

`macro_vintages` records when the panel was generated and how recent each
indicator's last observation is — 2025 CPI beside 2021 governance is not the same
evidence as all-2025. `vintage_scheme` is the field that matters later:
`"as-published-latest"` means these are latest published values, silently revised
by the World Bank over time. The point-in-time panel will write `"first-release"`
there, so ratings built on revised data can be excluded with a query rather than
a comment.

The module is pure — no database, no network, no clock — and the pipeline wraps
this call in a `try/except` that logs the traceback and passes `None`: provenance
is metadata, not the product, and a bug in it must never cost a country its
score.

In [21]:
input_manifest = provenance.build_input_manifest(
    items=items,
    prompt_entries=langchain_llm.prompt_entries(items),  # the very entries the prompt carried
    fulltext_ids=fulltext_ids,
    payload=payload,
    model_id=llm_output.get("model_id"),
    prompt_version=llm_output.get("prompt_version"),
    policy_version=llm_output.get("policy_version"),
    seed=ai_client.SEED,
)

vintage = input_manifest["macro_vintages"]
print(f"schema_version {input_manifest['schema_version']}   seed {input_manifest['seed']}   "
      f"git_sha {input_manifest['git_sha'] or '(GIT_SHA unset)'}")
print(f"macro panel: {vintage['vintage_scheme']}, from {vintage['panel_source']}, "
      f"generated {vintage['panel_generated_at']}, newest year {vintage['latest_year']}")
display(pd.Series(vintage["latest_year_by_indicator"], name="last observation").to_frame())

manifest_view = pd.DataFrame(input_manifest["articles"]).set_index("id")
for col in ("content_sha256", "prompt_text_sha256"):
    manifest_view[col] = manifest_view[col].str[:12]   # 12 hex chars is enough to eyeball
manifest_view[["source", "in_prompt", "in_fulltext", "content_chars",
               "content_sha256", "prompt_text_sha256"]]

schema_version 1   seed 42   git_sha (GIT_SHA unset)
macro panel: as-published-latest, from World Bank, generated 2026-07-28T20:15Z, newest year 2025


,last observation
Inflation (% y/y),2024
Unemployment (% labour force),2025
FDI inflow (% GDP),2025
Political stability (z-score),2024
Rule of law (z-score),2024
Income inequality (Gini),2024
GDP per-capita growth (% y/y),2025
Interest payments (% revenue),2024
"Political corruption index (0–1, higher = more corrupt)",2025


,source,in_prompt,in_fulltext,content_chars,content_sha256,prompt_text_sha256
id,,,,,,
a1,Breaking Defense,True,False,9560,4d5f6188296f,a7e7cf20a083
a2,Economy Middle East,True,False,10409,8a4bec73f932,d0aabcd2fd32
a3,Brookings,True,False,10020,681d005ac3b0,9b56dd5db8b9
a4,Yahoo Finance UK,True,False,2062,91e643a30366,fea3d11e4ea0
a5,Peter G. Peterson Foundation,True,False,3743,fe4b1855ccad,97663a516fda
a6,NBC News,True,False,6858,ea64a846f576,fdf8a911b89a
a7,Economy Middle East,True,False,8652,573596b98d68,a6ae3ac58ddd
a8,Foreign Policy,True,False,8037,a088e13c8650,f7cea58e73da
a9,FXStreet,True,False,5211,f1b645248516,595e16ec82d9


## Step 4 — Top-3 selection

The model groups articles covering the same underlying event into a shared
`topic_group`. With 3 or more distinct topics, the best article of each of the
top 3 topics wins — so the dashboard shows three *stories* rather than three
write-ups of one. With fewer topics, the remainder is backfilled by impact.

The table shows every candidate, not just the winners, so you can see what lost.

In [22]:
imp_map, topic_map = article_ranking.impact_topic_maps(llm_output)
items_by_id = {it.get("id"): it for it in items if isinstance(it, dict) and it.get("id")}
top_ids = article_ranking.select_top_ids(items_by_id, imp_map, topic_map, ISO2)

print(f"top 3: {top_ids}  (from {len(items_by_id)} candidates, "
      f"{len(set(topic_map.values()))} distinct topics)")
pd.DataFrame([
    {
        "id": aid,
        "selected": aid in top_ids,
        "impact": imp_map.get(aid),
        "topic_group": topic_map.get(aid),
        "published": it.get("published"),
        "title": it.get("title"),
    }
    for aid, it in items_by_id.items()
]).sort_values("impact", ascending=False).set_index("id")

2026-07-28 16:16:27,615 INFO    backend.utils.news_fetching.article_ranking: [US] AI identified 11 topics (used 1/article).
top 3: ['a12', 'a1', 'a3']  (from 20 candidates, 11 distinct topics)


,selected,impact,topic_group,published,title
id,,,,,
a10,False,0.85,us_iran_war,2026-07-26T15:17:00Z,United States could be in too deep in war with Iran to get out quickly - Our...
a18,False,0.85,us_iran_war,2026-07-28T01:26:12Z,Trump says US 'winning big time' against Tehran as Netanyahu prepares to unv...
a12,True,0.85,us_iran_war,2026-07-28T10:00:00Z,Jordan faces a dilemma in its growing role in the U.S.-Iran war - Los Angele...
a1,True,0.60,us_japan_missile_defense,2026-07-28T13:30:35Z,Time to decide: A critical moment in Japan-US missile defense cooperation - ...
a17,False,0.60,us_israel_relations,2026-07-28T03:51:00Z,Netanyahu Lands In DC Ahead Of Trump Meeting | LIVE BLOG - i24NEWS
a6,False,0.60,us_israel_relations,2026-07-28T16:39:00Z,Netanyahu joins Trump for high-stakes meeting at the White House - NBC News
a3,True,0.60,us_foreign_policy,2026-07-27T12:34:47Z,Why can’t the United States end allies’ wars? - Brookings
a2,False,0.40,us_fed_rate_decision,2026-07-28T12:37:57Z,Dollar holds four-week high at 101.425 as Fed hike odds near 40 percent - Ec...
a4,False,0.40,us_inflation_oil_prices,2026-07-26T15:11:03Z,Barclays Expects Earnings and Central Bank Decisions to Set the Market Tone ...


## Step 5 — images for the Top-3

Chosen articles still missing a thumbnail get one more try through Crawlbase,
which renders JavaScript. It costs a credit per call, hence the Top-3-only
scope, and no-ops entirely without `CRAWLBASE_TOKEN`.

In [23]:
before_images = {aid: items_by_id[aid].get("image") for aid in top_ids}
article_enrichment.enrich_top_images(top_ids, items_by_id)

pd.DataFrame([
    {"id": aid, "before": before_images[aid], "after": items_by_id[aid].get("image")}
    for aid in top_ids
]).set_index("id")

,before,after
id,,
a12,https://ca-times.brightspotcdn.com/dims4/default/812f6c6/2147483647/strip/tr...,https://ca-times.brightspotcdn.com/dims4/default/812f6c6/2147483647/strip/tr...
a1,https://breakingdefense.com/wp-content/uploads/sites/13/2026/07/japan-patrio...,https://breakingdefense.com/wp-content/uploads/sites/13/2026/07/japan-patrio...
a3,https://www.brookings.edu/wp-content/uploads/2026/07/GettyImages-2281061018....,https://www.brookings.edu/wp-content/uploads/2026/07/GettyImages-2281061018....


## Step 6 — the Top-3 rows

These are the rows that would be written to `risk_snapshot_article` and rendered
on the country page, followed by a rough preview of how they look there.

In [24]:
top_articles = article_ranking.build_top_articles(top_ids, items_by_id, imp_map)
pd.DataFrame(top_articles).set_index("rank")

,id,url,title,source,published_at,impact,summary,image
rank,,,,,,,,
1,a12,https://www.latimes.com/world-nation/story/2026-07-28/jordan-faces-dilemma-i...,Jordan faces a dilemma in its growing role in the U.S.-Iran war - Los Angele...,Los Angeles Times,2026-07-28T10:00:00Z,0.85,Jordan faces a dilemma in its growing role in the U.S.-Iran war - Click here...,https://ca-times.brightspotcdn.com/dims4/default/812f6c6/2147483647/strip/tr...
2,a1,https://breakingdefense.com/2026/07/time-to-decide-a-critical-moment-in-japa...,Time to decide: A critical moment in Japan-US missile defense cooperation - ...,Breaking Defense,2026-07-28T13:30:35Z,0.60,There is no more important bilateral air and missile defense cooperative rel...,https://breakingdefense.com/wp-content/uploads/sites/13/2026/07/japan-patrio...
3,a3,https://www.brookings.edu/articles/why-the-united-states-struggles-to-end-pa...,Why can’t the United States end allies’ wars? - Brookings,Brookings,2026-07-27T12:34:47Z,0.60,"Executive summary Wars, whether between or within states, are not ending in ...",https://www.brookings.edu/wp-content/uploads/2026/07/GettyImages-2281061018....


In [25]:
from IPython.display import HTML

HTML("".join(
    '<div style="display:flex;gap:12px;margin:12px 0;align-items:flex-start">'
    + (f'<img src="{a["image"]}" style="width:160px;border-radius:6px">' if a["image"] else "")
    + f'<div><b>#{a["rank"]} &middot; impact {a["impact"]}</b><br>'
      f'<a href="{a["url"]}" target="_blank">{a["title"]}</a><br>'
      f'<small>{a["source"]} &middot; {a["published_at"]}</small><br>'
      f'<small>{(a["summary"] or "")[:240]}</small></div></div>'
    for a in top_articles
))

## Step 7 — the snapshot payload (not written)

The pipeline would hand this dict to `data_push.upsert_snapshot`, which writes
`country`, `indicator`, `yearly_value`, `risk_snapshot`, and
`risk_snapshot_article`.

`risk_snapshot` keeps `score` and `bullet_summary` exactly as the front-end has
always read them, and stores everything from Steps 3b and 3c alongside: both
horizons raw and gated, the sub-factor detail, `condition_flags`, every article's
impact (not only the Top-3), `applied_rules`, `legal_gate`, the three version
stamps and `input_manifest`. Those columns are provisioned by an `ALTER TABLE ...
ADD COLUMN IF NOT EXISTS` the writer issues once per process — there is no
migration tool. On a re-run of the same `as_of`, `score` and `bullet_summary`
overwrite while every detail column `COALESCE`s, so a degraded re-run cannot blank
what a good one already stored.

**This notebook stops here on purpose** — no snapshot write, so a run can never
overwrite today's real snapshot for this country. (Step 2b's `article_digest`
rows are the one exception, and nothing reads those but Step 2b itself.) Use
`backend/tests/live_country_check.py` when you want the write plus verification
and cleanup.

In [26]:
snapshot = {**payload, "llm_output": llm_output, "top_articles": top_articles,
            "input_manifest": input_manifest}

# What upsert_snapshot validates before it would open a transaction.
print(f"country={snapshot['country']}  as_of<-{snapshot['_meta']['generated_at']}  "
      f"indicators={len(snapshot['indicators'])}  articles={len(snapshot['top_articles'])}  "
      f"manifest={len(input_manifest['articles'])} articles hashed")
print(json.dumps(snapshot, indent=2, default=str, ensure_ascii=False)[:3000])

country=US  as_of<-2026-07-28T20:15Z  indicators=9  articles=3  manifest=20 articles hashed
{
  "country": "US",
  "latest_year": 2025,
  "indicators": {
    "Inflation (% y/y)": {
      "latest": 2.95,
      "Δ1y": -1.167,
      "Δ5y": 1.137,
      "series": {
        "2015": 0.12,
        "2016": 1.26,
        "2017": 2.13,
        "2018": 2.44,
        "2019": 1.81,
        "2020": 1.23,
        "2021": 4.7,
        "2022": 8.0,
        "2023": 4.12,
        "2024": 2.95
      }
    },
    "Unemployment (% labour force)": {
      "latest": 4.2,
      "Δ1y": 0.176,
      "Δ5y": -3.857,
      "series": {
        "2016": 4.87,
        "2017": 4.36,
        "2018": 3.9,
        "2019": 3.67,
        "2020": 8.06,
        "2021": 5.35,
        "2022": 3.65,
        "2023": 3.64,
        "2024": 4.02,
        "2025": 4.2
      }
    },
    "FDI inflow (% GDP)": {
      "latest": 1.3,
      "Δ1y": 0.288,
      "Δ5y": 0.66,
      "series": {
        "2016": 2.52,
        "2017": 1.94,
     

## Summary

Everything the run produced, on one screen — the "did this make sense?" cell.

This is the whole of rating one country: `_process_country` from the macro panel
to the upsert, in order. What it leaves out are the other phases of `main.py`,
none of which touch a country's score — the economic-calendar and market-price
fetches, the IMF recent-indicator refresh, and the post-loop global alert
ranking, which pools every country's Top-3 and re-ranks them for the dashboard's
alert strip.

In [27]:
print(f"{NAME} ({ISO2})  -  risk score {llm_output.get('score')} (12m), "
      f"{llm_output.get('score_3m')} (3m)   as of {AS_OF}")
print(f"  ledgers        " + "  ".join(
    f"{k}={v}" for k, v in (llm_output.get("ledger_scores") or {}).items()))
print(f"  non_investable {llm_output.get('non_investable')}   "
      f"coverage {llm_output.get('evidence_coverage')}   "
      f"lint findings {len(findings)}")
print(f"  evidence       {sum(len(v) for k, v in evidence.items() if k.endswith('_inputs'))} "
      f"indicators, {len(evidence['computed'])} computed metrics, "
      f"~{len(json.dumps(evidence, ensure_ascii=False)) // 4} tokens")
print(f"  articles       {len(items)} fetched, {len(digested)} digested, "
      f"{len(fulltext_ids)} read in full, {len(top_articles)} shown")
print(f"  stamps         model {llm_output.get('model_id')}   "
      f"prompt {llm_output.get('prompt_version')}   "
      f"policy {llm_output.get('policy_version')}")

print()
print(llm_output.get("bullet_summary"))
print()
for a in top_articles:
    print(f"  #{a['rank']}  impact {a['impact']}  {a['title']}")
    print(f"      {a['source']} - {a['published_at']}")
    print(f"      {a['url']}")
    print()

United States (US)  -  risk score 0.62 (12m), 0.58 (3m)   as of 2026-07-28
  ledgers        friction=0.38  order_uncertainty=0.4  information_capacity=0.15  edge_vitality=0.6
  non_investable False   coverage 0.75   lint findings 0
  evidence       20 indicators, 9 computed metrics, ~1664 tokens
  articles       20 fetched, 20 digested, 3 read in full, 3 shown
  stamps         model gpt-4o-2024-08-06   prompt v3.1   policy p2.0-observe-only

The U.S. faces moderate friction due to rising interest payments and a low tax revenue to GDP ratio. Order uncertainty is moderate, influenced by inflation and political stability concerns. Information capacity is strong, with reliable statistics. The ongoing U.S.-Iran war significantly impacts risk, raising short-term investor concerns. Edge vitality is moderate, reflecting adaptive capacity. Coverage is robust but some indicators are stale.

  #1  impact 0.85  Jordan faces a dilemma in its growing role in the U.S.-Iran war - Los Angeles Times
   